In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [2]:
ROOT = Path("/Users/kirik/Desktop/Pythom/LK317/RiskDecOfPortfolio")
DATA_RAW = ROOT / "data" / "raw"

START_DATE = "2003-09-20"
END_DATE   = "2024-12-31"

In [3]:
spy_agg = pd.read_csv(DATA_RAW / "SPYAGG.csv", parse_dates=["date"])
spy_agg = spy_agg.set_index("date").sort_index()

spy_agg.head()

,PERMNO,PRC,VOL,RET
date,,,,
2000-01-03,84398,145.4375,8164299,-0.009787
2000-01-04,84398,139.7500,8089799,-0.039106
2000-01-05,84398,140.0000,12177899,0.001789
2000-01-06,84398,137.7500,6227199,-0.016071
2000-01-07,84398,145.7500,8066599,0.058076


In [4]:
spy_agg.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 11641 entries, 2000-01-03 to 2024-12-31
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   PERMNO  11641 non-null  int64  
 1   PRC     11641 non-null  float64
 2   VOL     11641 non-null  int64  
 3   RET     11641 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 454.7+ KB


In [5]:
spy_agg.index.min(), spy_agg.index.max()

(Timestamp('2000-01-03 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [29]:
spy_agg["PERMNO"].nunique(), spy_agg["PERMNO"].value_counts().head()

(2,
 PERMNO
 84398    6289
 89848    5352
 Name: count, dtype: int64)

In [30]:
spy_agg["RET"] = pd.to_numeric(spy_agg["RET"], errors="coerce")
spy_agg["RET"].isna().sum()

/var/folders/y1/t79wmh0x3zvfvmjwgtpyh9h40000gn/T/ipykernel_52558/1355676606.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  spy_agg["RET"] = pd.to_numeric(spy_agg["RET"], errors="coerce")


np.int64(1)

In [31]:
wide = spy_agg[["PERMNO", "RET"]].reset_index().pivot(index="date", columns="PERMNO", values="RET")
wide.head()

PERMNO,84398,89848
date,,
2000-01-03,-0.009787,NaN
2000-01-04,-0.039106,NaN
2000-01-05,0.001789,NaN
2000-01-06,-0.016071,NaN
2000-01-07,0.058076,NaN


In [32]:
starts = wide.apply(lambda s: s.first_valid_index())
starts.sort_values()

PERMNO
84398   2000-01-03
89848   2003-09-29
dtype: datetime64[ns]

In [ ]:
permno_spy = starts.idxmin()            
permno_agg = starts.idxmax()            

ret = wide.rename(columns={permno_spy: "SPY_RET", permno_agg: "AGG_RET"})
ret = ret.loc[START_DATE:END_DATE]
ret.head()

PERMNO,SPY_RET,AGG_RET
date,,
2003-09-22,-0.010803,NaN
2003-09-23,0.003803,NaN
2003-09-24,-0.017777,NaN
2003-09-25,-0.008209,NaN
2003-09-26,-0.003291,NaN


In [6]:
rf = pd.read_csv(DATA_RAW / "RF.csv", parse_dates=["date"])
rf = rf.set_index("date").sort_index()

rf.head()

,mktrf,smb,hml,rf,umd
date,,,,,
2000-01-03,-0.0071,0.0055,-0.0131,0.0002,-0.0006
2000-01-04,-0.0406,-0.0002,0.0207,0.0002,-0.0191
2000-01-05,-0.0009,0.0031,-0.0005,0.0002,-0.0049
2000-01-06,-0.0074,-0.0043,0.0124,0.0002,-0.0149
2000-01-07,0.0321,-0.0040,-0.0157,0.0002,0.0059


In [7]:
rf.describe()

,mktrf,smb,hml,rf,umd
count,6539.000000,6539.000000,6539.000000,6539.000000,6539.000000
mean,0.000319,0.000042,0.000101,0.000075,0.000125
std,0.012387,0.006435,0.007799,0.000085,0.010646
min,-0.120100,-0.044800,-0.050300,0.000000,-0.143700
25%,-0.004900,-0.003700,-0.003300,0.000000,-0.004400
50%,0.000700,0.000100,-0.000100,0.000000,0.000600
75%,0.006100,0.003700,0.003400,0.000100,0.005300
max,0.113600,0.054500,0.067300,0.000300,0.071400


In [34]:
ff = rf.loc[START_DATE:END_DATE].copy()

In [26]:
pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("/Users/kirik/Desktop/Pythom/LK317/RiskDecOfPortfolio")
DATA_RAW = ROOT / "data" / "raw"

ind = pd.read_excel(DATA_RAW / "49.xlsx")

#Missing data are indicated by -99.99 or -999.
#Copyright 2025 Eugene F. Fama and Kenneth R. French

ind.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Agric,Food,Soda,Beer,Smoke,Toys,Fun,Books,Hshld,Clths,Hlth,MedEq,Drugs,Chems,Rubbr,Txtls,BldMt,Cnstr,Steel,FabPr,Mach,ElcEq,Autos,Aero,Ships,Guns,Gold,Mines,Coal,Oil,Util,Telcm,PerSv,BusSv,Hardw,Softw,Chips,LabEq,Paper,Boxes,Trans,Whlsl,Rtail,Meals,Banks,Insur,RlEst,Fin,Other
4,19990604,0.73,-0.11,-0.05,1.52,0.45,0.88,0.41,1.08,0.45,2.14,0.81,0.62,1,0.39,-0.46,-0.23,-0.07,-0.32,0.86,-0.41,0.04,0.1,0.29,2.02,1.57,-0.65,-0.33,0.59,-0.21,0.72,0.49,1.54,-0.23,0.84,1.18,1.26,1.77,1.06,0.89,1.5,0.21,0.6,0.82,0.5,0.46,-0.06,-0.47,0.08,1.9


In [39]:
import pandas as pd
import numpy as np

raw_x = pd.read_excel(DATA_RAW / "49.xlsx", header=None)

# Header row is row 3, data starts at row 4
headers = raw_x.iloc[3].tolist()

# Force first column name
headers[0] = "date"

data = raw_x.iloc[4:].copy()
data.columns = headers

In [40]:
def dedupe_columns(cols):
    seen = {}
    out = []
    for c in cols:
        c = str(c)
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}.{seen[c]}")
    return out

data.columns = dedupe_columns(data.columns)

In [41]:
data["date"] = pd.to_datetime(
    data["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

data = (
    data
    .dropna(subset=["date"])
    .set_index("date")
    .sort_index()
)

/var/folders/y1/t79wmh0x3zvfvmjwgtpyh9h40000gn/T/ipykernel_52558/3416227633.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data["date"] = pd.to_datetime(


In [42]:
data = data.apply(pd.to_numeric, errors="coerce")

In [43]:
ind = data.replace([-99.99, -999], np.nan)
ind = ind.loc[START_DATE:END_DATE]

In [44]:
ind.columns[:10]
ind.shape
ind.index.min(), ind.index.max()

(Timestamp('2003-09-22 00:00:00'), Timestamp('2024-12-31 00:00:00'))

In [45]:
ind.std().median()

np.float64(1.5666835443212443)

sector_map = {
    "Tech": ["Softw", "Chips"],
    "Financials": ["Banks", "Insur", "Fin"],
    "Energy": ["Oil", "Coal", "Mines"],
    "Industrials": ["Mach", "Trans", "Aero"],
    "Consumer": ["Food", "Rtail", "Meals"],
    "Healthcare": ["Hlth", "MedEq", "Drugs"],
}

In [21]:
vix = pd.read_csv(DATA_RAW / "VIX.csv", parse_dates=["Date"])
vix = vix.set_index("Date").sort_index()
vix = vix.loc[START_DATE:END_DATE]

vix.head()


,VIX,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
Date,,,,,,
2003-09-22,19.65,NaN,NaN,NaN,NaN,NaN
2003-09-22,19.65,NaN,NaN,NaN,NaN,NaN
2003-09-23,19.47,NaN,NaN,NaN,NaN,NaN
2003-09-24,21.22,NaN,NaN,NaN,NaN,NaN
2003-09-25,22.26,NaN,NaN,NaN,NaN,NaN


In [22]:
vix.describe()

,VIX,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
count,5378.000000,0.0,0.0,0.0,0.0,0.0
mean,18.958109,NaN,NaN,NaN,NaN,NaN
std,8.552206,NaN,NaN,NaN,NaN,NaN
min,9.140000,NaN,NaN,NaN,NaN,NaN
25%,13.420000,NaN,NaN,NaN,NaN,NaN
50%,16.505000,NaN,NaN,NaN,NaN,NaN
75%,21.717500,NaN,NaN,NaN,NaN,NaN
max,82.690000,NaN,NaN,NaN,NaN,NaN


In [ ]:
vix = pd.read_csv(DATA_RAW / "VIX.csv", parse_dates=["Date"])
vix = vix[["Date", "VIX"]]  
vix = vix.dropna(subset=["VIX"])
vix = vix.drop_duplicates(subset=["Date"], keep="last")  
vix = vix.set_index("Date").sort_index()
vix = vix.loc[START_DATE:END_DATE]
vix.head()

,VIX
Date,
2003-09-22,19.65
2003-09-23,19.47
2003-09-24,21.22
2003-09-25,22.26
2003-09-26,22.23
